# Herança entre Classes

- Conceito básico de Orientação a Objetos:
    - Uma classe pode ser "filha" de outra.
    - Define um subtipo da classe "mãe".
        - Herda todas as suas características (propriedades, métodos).
        - Mas pode adicionar e/ou alterar coisas.
- Exemplo de um Sistema Académico:
    - Toda `Pessoa` tem um `nome`, `numero` e `idade`.
        - Idade pode ser incrementada.
    - Um `Funcionario` é uma `Pessoa`, mas tem um `ordenado`.
    - Um `Aluno` é uma pessoa (mas não um `Funcionario`), e tem um `curso`.
    - Um `Professor` e um tipo particular de `Funcionario`.

![img/HierarquiaPessoa.png](img/HierarquiaPessoa.png)

# Herança em Kotlin: Classe `Pessoa`

In [3]:
open class Pessoa(val nome: String, val numero: Int, var idade: Int) {
    fun incrementaIdade() { idade = idade + 1 }
}

- **Notas**:
    - `open`: marca que a classe pode ser estendida / herdada.
        - Opõe-se ao `final`.
        - Classes são `final` por omissão.  

# Herança em Kotlin: Classes `Funcionário` e `Aluno`

In [4]:
open class Funcionario(nome: String, numero: Int, idade: Int, var ordenado: Double): Pessoa(nome, numero, idade) 
class Aluno(nome: String, numero: Int, idade: Int, var curso: String): Pessoa(nome, numero, idade) 

- **Notas**:
    - `: Pessoa(...)`: denota que classes estendem `Pessoa`.
        - Também especifica que construtor de `Pessoa` queremos chamar.
    - `nome`, `numero` e `idade` declarados como parâmetros do construtor, mas não como propriedades.
        - Já existem em `Pessoa`, serão definidas no construtor de `Pessoa`. 
    - `Funcionário` declarada como `open`.
        - Será estendida por `Professor`.

# Herança em Kotlin: Classe `Professor`

In [5]:
class Professor(nome: String, numero: Int, idade: Int, ordenado: Double): Funcionario(nome, numero, idade, ordenado)

- **Notas**:
    - Herda de `Funcionário`.
        - Transitivamente, herda de `Pessoa`.  

# Herança em Kotlin: Instanciação e Uso

In [7]:
val aluno1 = Aluno("João", 12345, 19, "LEIC")
val professor1 = Professor("Paulo", 54321, 40, 1000.0)
println(aluno1.idade)      // 19
println(professor1.idade)  // 40
professor1.incrementaIdade()
println(professor1.idade)  // 41

19
40
41


- **Notas**:
    - `.incrementaIdade()` definida em `Pessoa`, mas herdada por `Professor`.

# Herança em Kotlin: Sobrescrita de Métodos

- Suponha um método `.descricao()` em `Pessoa`.
    - Descreve aquela pessoa na organização.

In [11]:
open class Pessoa(val nome: String, val numero: Int, var idade: Int) {
    open fun descricao() = "${nome} tem número ${numero}"
    fun incrementaIdade() { idade = idade + 1 }
}

- Em `Pessoa`, descrição é genérica.
- Gostaríamos de ter descrições mais específicas, a depender do subtipo de `Pessoa`.
- Cada subtipo deve poder sobrescrever `.descricao()` de acordo com as suas especificidades.
    - Marcamos o método como `open`. 

# Herança em Kotlin: Sobrescrita de Métodos (II)

- Podemos sobrescrever `.descricao()` em `Aluno`.

In [13]:
class Aluno(nome: String, numero: Int, idade: Int, var curso: String): Pessoa(nome, numero, idade) {
    override fun descricao() = "${nome} é aluno e tem número ${numero}"
}

- **Notas**:
    - `override`: modificador necessário quando queremos sobrescrever método herdado.

# Herança em Kotlin: Sobrescrita de Métodos (III)

- Sobrescrever em `Funcionario` e `Professor`

In [18]:
open class Funcionario(nome: String, numero: Int, idade: Int, var ordenado: Double): Pessoa(nome, numero, idade) {
    override fun descricao() = "${nome} é funcionário e tem número ${numero}"
}
class Professor(nome: String, numero: Int, idade: Int, ordenado: Double): Funcionario(nome, numero, idade, ordenado) {
    override fun descricao() = "${nome} é professor e tem número ${numero}"
}
println(Professor("Paulo", 54321, 40, 1000.0).descricao())

Paulo é professor e tem número 54321


# Polimorfismo

- Capacidade de tratar objetos de tipos diferentes por uma **interface comum**.
    - Mas cada objeto executa o **comportamento apropriado para a sua própria classe**.
- Nos exemplos anteriores:
    - `Professor`, `Funcionario`, `Aluno` tem interface comum `Pessoa`.
        - Todos podem ser tratados como instâncias de `Pessoa`.
    - Mas cada um tem sua versão de `.descricao()`.

In [26]:
val listaPessoas: List<Pessoa> = listOf(Professor("Paulo", 54321, 40, 1000.0), 
                          Aluno("João", 12345, 19, "LEIC"),
                          Funcionario("Cristina", 23456, 35, 2000.0),
                          Pessoa("Rodrigo", 65432, 27))
listaPessoas.forEach {
    println(it.descricao())   // Chamada polimórfica: método chamado de acordo com subtipo de Pessoa
}

Paulo é professor e tem número 54321
João é aluno e tem número 12345
Cristina é funcionário e tem número 23456
Rodrigo tem número 65432


# Polimorfismo e Funções de Extensão

- Funções de extensão são resolvidas em **tempo de compilação**.
    - **Não há polimorfismo**.
- Suponha que `.descricao()` de `Professor` é definido como função de extensão, ao invés de método:

In [28]:
class Professor(nome: String, numero: Int, idade: Int, ordenado: Double): Funcionario(nome, numero, idade, ordenado)
fun Professor.descricao() = "${nome} é professor e tem número ${numero}"
val listaPessoas: List<Pessoa> = listOf(Professor("Paulo", 54321, 40, 1000.0), 
                          Aluno("João", 12345, 19, "LEIC"),
                          Funcionario("Cristina", 23456, 35, 2000.0),
                          Pessoa("Rodrigo", 65432, 27))
listaPessoas.forEach {
    println(it.descricao())   // Chamada polimórfica: método chamado de acordo com subtipo de Pessoa
}                             // Mase, em tempo de execução, Professor.descricao() aponta para Funcionario.descricao().

Paulo é funcionário e tem número 54321
João é aluno e tem número 12345
Cristina é funcionário e tem número 23456
Rodrigo tem número 65432


# Classes Abstratas

- Algumas vezes, não faz sentido que a classe base seja **instanciável**.
    - Ela serve apenas como um modelo para as subclasses.
    - *e.g.*, pode não fazer sentido instanciar uma `Pessoa` sem um subtipo específico.
- **Solução**: classe base abstrata.

In [ ]:
abstract class Pessoa(val nome: String, val numero: Int, var idade: Int) {
    abstract fun descricao(): String
    fun incrementaIdade() { idade = idade + 1 }
}
//val pessoa1 = Pessoa("Rodrigo", 65432, 27) // Erro: não se pode instanciar uma classe abstrata!

- **Notas**:
    - Classe abstrata pode ter mistura de métodos com e sem implementação.
        - Sem implementação: métodos abstratos, subclasses são obrigadas a implementá-los.
        - Com implementação: por omissão, finais, mas podem ser marcados como `open`. 